# Backbone Experiment: vit_b_16

## Setup

In [2]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))  # repo root
from config import Config
from data.cifake import get_cifake_loaders
from experiments.train import train
from experiments.evaluate import full_evaluation
from experiments.baseline_spatial_only import run_spatial_only_baseline
from tqdm.notebook import tqdm
import pandas as pd
import torch

print(f"Device: { 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'}")
if torch.cuda.is_available():
    print(f"GPU:    {torch.cuda.get_device_name(0)}")
    print(f"VRAM:   {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


/Users/chauc/PycharmProjects/asfr/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: mps


## Shared Config

In [3]:
# Only line to change
BACKBONE = "vit_b_16"

def make_cfg(fusion_mode, frozen=False):
    cfg = Config()
    cfg.data.cifake_root    = "../data/raw/cifake"
    cfg.data.num_workers    = 4 # change to 0 if on Windows
    cfg.train.backbone_lr   = 1e-4  
    cfg.train.lr            = 1e-4
    cfg.data.batch_size     = 64
    cfg.data.jpeg_aug       = True
    cfg.backbone.name       = BACKBONE
    cfg.backbone.pretrained = True
    cfg.backbone.frozen     = frozen
    cfg.backbone.img_size   = cfg.data.image_size  
    cfg.fusion.mode         = fusion_mode
    cfg.train.epochs        = 50
    cfg.experiment_name     = f"{BACKBONE}_{fusion_mode}{'_frozen' if frozen else ''}"
    cfg.notes               = f"CIFAKE · {BACKBONE} · {fusion_mode}{'_frozen' if frozen else ''}"
    return cfg

# Load data
cfg_data = make_cfg("joint_only") # backbone settings only, fusion mode ignored
train_loader, val_loader, test_loader = get_cifake_loaders(cfg_data)
print(f"Train: {len(train_loader.dataset):,}  Val: {len(val_loader.dataset):,}  Test: {len(test_loader.dataset):,}")

Train: 85,000  Val: 15,000  Test: 20,000
Train: 85,000  Val: 15,000  Test: 20,000


## Experiment 0: Spatial-Only Baseline
Trains only the spatial backbone as a standalone classifier with no frequency branch.
This gives the honest floor. How well the backbone alone can classify real vs fake.
All delta values in later experiments are relative to this number.


In [3]:
cfg0 = make_cfg("joint_only")  # backbone settings only, fusion mode ignored
cfg0.experiment_name = f"{BACKBONE}_spatial_only"
cfg0.notes           = f"spatial-only baseline, no freq branch, {BACKBONE}"
spatial_acc = run_spatial_only_baseline(cfg0, train_loader, val_loader, test_loader)
print(f"\nSpatial-only floor: {spatial_acc:.1%}")
print("All subsequent delta values are relative to this number.")

Device: mps
Training spatial-only baseline (vit_b_16) for 50 epochs...
Train: 85,000  Val: 15,000


Epoch   1/50 | train_loss=0.3131 | val_acc=90.9% | best=0.0% | patience=1/5
  ✓ New best saved (90.9%)


Epoch   2/50 | train_loss=0.2277 | val_acc=92.7% | best=90.9% | patience=1/5
  ✓ New best saved (92.7%)


Epoch   3/50 | train_loss=0.2010 | val_acc=92.6% | best=92.7% | patience=1/5


Epoch   4/50 | train_loss=0.1849 | val_acc=94.1% | best=92.7% | patience=2/5
  ✓ New best saved (94.1%)


...

Epoch  27/50 | train_loss=0.0504 | val_acc=96.4% | best=96.1% | patience=3/5
  ✓ New best saved (96.4%)


Epoch  28/50 | train_loss=0.0442 | val_acc=96.2% | best=96.4% | patience=1/5


Epoch  29/50 | train_loss=0.0426 | val_acc=96.3% | best=96.4% | patience=2/5


Epoch  30/50 | train_loss=0.0383 | val_acc=96.1% | best=96.4% | patience=3/5


Epoch  31/50 | train_loss=0.0360 | val_acc=96.4% | best=96.4% | patience=4/5
  Early stopping at epoch 31 — best val_acc=96.4%

Spatial-only results (vit_b_16):
  Accuracy: 96.3%
  AUC-ROC:  0.994
  F1:       0.963
Results saved → ./results/results.csv  (vit_b_16_spatial_only, acc=0.9629)

Spatial-only floor: 96.3%
All subsequent delta values are relative to this number.


## Experiment 1: Joint-Only Baseline
Both branches concatenated into a single shared classifier. No weighting between branches.
This is the floor — all other experiments are compared against this delta value.


In [4]:
cfg1 = make_cfg("joint_only")
print(f"Running: {cfg1.experiment_name}")
train(cfg1, train_loader, val_loader, test_loader)

Running: vit_b_16_joint_only
Device: mps
torch.compile skipped — device is mps

Experiment: vit_b_16_joint_only
Backbone: vit_b_16 | Fusion: joint_only | Frozen: False
Epochs: 50 | LR: 0.0001 | Batch: 64



Epoch   1/50 | train_loss=0.5589 | val_acc=91.0% | val_auc=0.980 | val_f1=0.904 | best=0.0% | patience=1/5
  grad norms — freq=0.7670 | spatial=0.6262
  -> Saved best val_acc=91.0%


Epoch   2/50 | train_loss=0.4265 | val_acc=94.4% | val_auc=0.988 | val_f1=0.945 | best=91.0% | patience=0/5
  -> Saved best val_acc=94.4%


Epoch   3/50 | train_loss=0.3904 | val_acc=93.0% | val_auc=0.990 | val_f1=0.934 | best=94.4% | patience=0/5


Epoch   4/50 | train_loss=0.3602 | val_acc=94.6% | val_auc=0.991 | val_f1=0.948 | best=94.4% | patience=1/5
  -> Saved best val_acc=94.6%


...

Epoch  20/50 | train_loss=0.1930 | val_acc=96.8% | val_auc=0.995 | val_f1=0.968 | best=96.9% | patience=1/5


Epoch  21/50 | train_loss=0.1901 | val_acc=96.8% | val_auc=0.995 | val_f1=0.968 | best=96.9% | patience=2/5
  grad norms — freq=0.9960 | spatial=0.0881


Epoch  22/50 | train_loss=0.1835 | val_acc=96.9% | val_auc=0.995 | val_f1=0.969 | best=96.9% | patience=3/5


Epoch  23/50 | train_loss=0.1755 | val_acc=96.9% | val_auc=0.995 | val_f1=0.970 | best=96.9% | patience=4/5
  Early stopping at epoch 23 — best val_acc=96.9%

Training complete. Best val accuracy: 96.9%
Results will be logged to CSV after full_evaluation().


0.969

In [5]:
results1 = full_evaluation(
    cfg1,
    checkpoint_path=f"./checkpoints/best_{cfg1.experiment_name}.pt",
    test_loader=test_loader,
)

Loaded checkpoint: ./checkpoints/best_vit_b_16_joint_only.pt

EVALUATION — vit_b_16_joint_only
Backbone: vit_b_16 | Fusion: joint_only | Frozen: False
  Joint accuracy:     96.8%
  AUC-ROC:            0.995
  F1:                 0.968
  Spatial-only:       95.1%
  Freq-only:          91.2%
  Delta (Δ):          +1.7%  (freq branch contribution)

  No warning signs triggered.
Results saved → ./results/results.csv  (vit_b_16_joint_only, acc=0.9681)


## Experiment 2: Scalar Fusion
Two learned scalars (a, b) weight each branch. Softmax ensures a+b=1 always.
Watch the scalar values logged each epoch. `b`is the frequency weight.
If `b` drops below 0.1 early in training the frequency branch is being ignored.


In [6]:
cfg2 = make_cfg("scalar")
print(f"Running: {cfg2.experiment_name}")
train(cfg2, train_loader, val_loader, test_loader)

Running: vit_b_16_scalar
Device: mps


torch.compile skipped — device is mps

Experiment: vit_b_16_scalar
Backbone: vit_b_16 | Fusion: scalar | Frozen: False
Epochs: 50 | LR: 0.0001 | Batch: 64



Epoch   1/50 | train_loss=0.5535 | val_acc=93.6% | val_auc=0.983 | val_f1=0.937 | best=0.0% | patience=1/5
  scalars — spatial=0.492 freq=0.508
  grad norms — freq=0.4646 | spatial=0.8836
  -> Saved best val_acc=93.6%


Epoch   2/50 | train_loss=0.4253 | val_acc=93.7% | val_auc=0.986 | val_f1=0.935 | best=93.6% | patience=0/5
  scalars — spatial=0.491 freq=0.509


Epoch   3/50 | train_loss=0.3948 | val_acc=94.4% | val_auc=0.987 | val_f1=0.944 | best=93.6% | patience=1/5
  scalars — spatial=0.490 freq=0.510
  -> Saved best val_acc=94.4%


...

Epoch  10/50 | train_loss=0.2835 | val_acc=95.8% | val_auc=0.993 | val_f1=0.959 | best=96.0% | patience=1/5
  scalars — spatial=0.486 freq=0.514


Epoch  11/50 | train_loss=0.2688 | val_acc=95.5% | val_auc=0.993 | val_f1=0.956 | best=96.0% | patience=2/5
  scalars — spatial=0.486 freq=0.514
  grad norms — freq=0.8484 | spatial=0.5262


Epoch  12/50 | train_loss=0.2635 | val_acc=95.9% | val_auc=0.993 | val_f1=0.960 | best=96.0% | patience=3/5
  scalars — spatial=0.486 freq=0.514


Epoch  13/50 | train_loss=0.2535 | val_acc=96.0% | val_auc=0.994 | val_f1=0.960 | best=96.0% | patience=4/5
  scalars — spatial=0.485 freq=0.515
  Early stopping at epoch 13 — best val_acc=96.0%

Training complete. Best val accuracy: 96.0%
Results will be logged to CSV after full_evaluation().


0.9604666666666667

In [7]:
results2 = full_evaluation(
    cfg2,
    checkpoint_path=f"./checkpoints/best_{cfg2.experiment_name}.pt",
    test_loader=test_loader,
)

Loaded checkpoint: ./checkpoints/best_vit_b_16_scalar.pt

EVALUATION — vit_b_16_scalar
Backbone: vit_b_16 | Fusion: scalar | Frozen: False
  Joint accuracy:     95.8%
  AUC-ROC:            0.993
  F1:                 0.958
  Spatial-only:       93.4%
  Freq-only:          89.1%
  Delta (Δ):          +2.4%  (freq branch contribution)

  No warning signs triggered.
Results saved → ./results/results.csv  (vit_b_16_scalar, acc=0.958)


## Experiment 3: Gating Fusion 
A small MLP outputs a per-image gate value in [0,1] controlling how much to trust the frequency branch.
Key metric beyond accuracy: **gate entropy must be > 0.3 nats**.
Below that the gate is outputting near-constant values and is not genuinely adapting per sample.


In [8]:
cfg3 = make_cfg("gating")
print(f"Running: {cfg3.experiment_name}")
train(cfg3, train_loader, val_loader, test_loader)

Running: vit_b_16_gating
Device: mps
torch.compile skipped — device is mps

Experiment: vit_b_16_gating
Backbone: vit_b_16 | Fusion: gating | Frozen: False
Epochs: 50 | LR: 0.0001 | Batch: 64



Epoch   1/50 | train_loss=0.6354 | val_acc=93.5% | val_auc=0.984 | val_f1=0.936 | best=0.0% | patience=1/5
  gate — entropy=1.100 nats | mean=0.619 | var=0.0015
  grad norms — freq=0.4293 | spatial=0.8990
  -> Saved best val_acc=93.5%


Epoch   2/50 | train_loss=0.5156 | val_acc=93.1% | val_auc=0.987 | val_f1=0.934 | best=93.5% | patience=0/5
  gate — entropy=1.321 nats | mean=0.585 | var=0.0020


Epoch   3/50 | train_loss=0.4761 | val_acc=92.8% | val_auc=0.989 | val_f1=0.932 | best=93.5% | patience=1/5
  gate — entropy=1.428 nats | mean=0.584 | var=0.0029


Epoch   4/50 | train_loss=0.4572 | val_acc=95.2% | val_auc=0.990 | val_f1=0.953 | best=93.5% | patience=2/5
  gate — entropy=0.863 nats | mean=0.592 | var=0.0006
  -> Saved best val_acc=95.2%


...

Epoch  15/50 | train_loss=0.3077 | val_acc=96.4% | val_auc=0.995 | val_f1=0.964 | best=96.5% | patience=1/5
  gate — entropy=1.163 nats | mean=0.536 | var=0.0013


Epoch  16/50 | train_loss=0.2966 | val_acc=96.4% | val_auc=0.995 | val_f1=0.964 | best=96.5% | patience=2/5
  gate — entropy=1.311 nats | mean=0.533 | var=0.0018
  grad norms — freq=0.7495 | spatial=0.6616


Epoch  17/50 | train_loss=0.3008 | val_acc=96.5% | val_auc=0.994 | val_f1=0.965 | best=96.5% | patience=3/5
  gate — entropy=1.413 nats | mean=0.573 | var=0.0026


Epoch  18/50 | train_loss=0.2881 | val_acc=96.2% | val_auc=0.995 | val_f1=0.963 | best=96.5% | patience=4/5
  gate — entropy=1.170 nats | mean=0.558 | var=0.0014
  Early stopping at epoch 18 — best val_acc=96.5%

Training complete. Best val accuracy: 96.5%
Results will be logged to CSV after full_evaluation().


0.9649333333333333

In [9]:
results3 = full_evaluation(
    cfg3,
    checkpoint_path=f"./checkpoints/best_{cfg3.experiment_name}.pt",
    test_loader=test_loader,
)

Loaded checkpoint: ./checkpoints/best_vit_b_16_gating.pt

EVALUATION — vit_b_16_gating
Backbone: vit_b_16 | Fusion: gating | Frozen: False
  Joint accuracy:     96.7%
  AUC-ROC:            0.995
  F1:                 0.967
  Spatial-only:       94.6%
  Freq-only:          90.1%
  Delta (Δ):          +2.1%  (freq branch contribution)

  Gate distribution:
    entropy:  1.499 nats  (OK)
    mean:     0.555
    variance: 0.0033

  No warning signs triggered.
Results saved → ./results/results.csv  (vit_b_16_gating, acc=0.9673)


## Experiment 4: Gating Fusion, Frozen Backbone
Same as Experiment 3 but backbone weights are locked.
Only the projection head, frequency branch, fusion, and joint head train.

If the frequency branch helps more here than in Experiment 3, it means the backbone
was learning to capture spectral information during fine-tuning in Experiment 3, essentially teaching itself what the frequency branch provides.


In [10]:
cfg4 = make_cfg("gating", frozen=True)
print(f"Running: {cfg4.experiment_name}")
train(cfg4, train_loader, val_loader, test_loader)

Running: vit_b_16_gating_frozen
Device: mps
torch.compile skipped — device is mps

Experiment: vit_b_16_gating_frozen
Backbone: vit_b_16 | Fusion: gating | Frozen: True
Epochs: 50 | LR: 0.0001 | Batch: 64



Epoch   1/50 | train_loss=0.7194 | val_acc=86.2% | val_auc=0.952 | val_f1=0.852 | best=0.0% | patience=1/5
  gate — entropy=1.707 nats | mean=0.519 | var=0.0053
  grad norms — freq=0.9297 | spatial=0.3568
  -> Saved best val_acc=86.2%


Epoch   2/50 | train_loss=0.5990 | val_acc=89.9% | val_auc=0.964 | val_f1=0.902 | best=86.2% | patience=0/5
  gate — entropy=1.724 nats | mean=0.584 | var=0.0048
  -> Saved best val_acc=89.9%


Epoch   3/50 | train_loss=0.5496 | val_acc=89.8% | val_auc=0.965 | val_f1=0.901 | best=89.9% | patience=0/5
  gate — entropy=2.042 nats | mean=0.605 | var=0.0087


Epoch   4/50 | train_loss=0.5234 | val_acc=89.5% | val_auc=0.969 | val_f1=0.902 | best=89.9% | patience=1/5
  gate — entropy=1.982 nats | mean=0.624 | var=0.0077


...

Epoch  19/50 | train_loss=0.3920 | val_acc=90.0% | val_auc=0.979 | val_f1=0.907 | best=92.7% | patience=1/5
  gate — entropy=2.438 nats | mean=0.722 | var=0.0220


Epoch  20/50 | train_loss=0.3913 | val_acc=92.0% | val_auc=0.979 | val_f1=0.923 | best=92.7% | patience=2/5
  gate — entropy=2.437 nats | mean=0.722 | var=0.0220


Epoch  21/50 | train_loss=0.3869 | val_acc=91.6% | val_auc=0.979 | val_f1=0.920 | best=92.7% | patience=3/5
  gate — entropy=2.400 nats | mean=0.720 | var=0.0199
  grad norms — freq=0.9515 | spatial=0.2420


Epoch  22/50 | train_loss=0.3824 | val_acc=92.2% | val_auc=0.980 | val_f1=0.926 | best=92.7% | patience=4/5
  gate — entropy=2.527 nats | mean=0.705 | var=0.0275
  Early stopping at epoch 22 — best val_acc=92.7%

Training complete. Best val accuracy: 92.7%
Results will be logged to CSV after full_evaluation().


0.9268

In [11]:
results4 = full_evaluation(
    cfg4,
    checkpoint_path=f"./checkpoints/best_{cfg4.experiment_name}.pt",
    test_loader=test_loader,
)

Loaded checkpoint: ./checkpoints/best_vit_b_16_gating_frozen.pt

EVALUATION — vit_b_16_gating_frozen
Backbone: vit_b_16 | Fusion: gating | Frozen: True
  Joint accuracy:     92.9%
  AUC-ROC:            0.980
  F1:                 0.931
  Spatial-only:       82.0%
  Freq-only:          90.4%
  Delta (Δ):          +10.9%  (freq branch contribution)

  Gate distribution:
    entropy:  2.435 nats  (OK)
    mean:     0.696
    variance: 0.0209

  No warning signs triggered.
Results saved → ./results/results.csv  (vit_b_16_gating_frozen, acc=0.9292)


## Summary: vit_b_16
Compare all four runs. Key columns: accuracy, delta (freq branch contribution), gate_entropy.


In [12]:
df = pd.read_csv("./results/results.csv")
mask = df["backbone"] == BACKBONE
cols = ["experiment_name", "fusion", "frozen", "accuracy", "auc_roc", "f1", "gate_entropy"]
print(df[mask][cols].to_string(index=False))

       experiment_name     fusion  frozen  accuracy  auc_roc     f1  gate_entropy
 vit_b_16_spatial_only joint_only   False    0.9629   0.9942 0.9632        0.0000
   vit_b_16_joint_only joint_only   False    0.9681   0.9953 0.9682        0.0000
       vit_b_16_scalar     scalar   False    0.9580   0.9929 0.9583        0.0000
       vit_b_16_gating     gating   False    0.9673   0.9947 0.9674        1.4990
vit_b_16_gating_frozen     gating    True    0.9292   0.9805 0.9308        2.4346


**What to look for:**
- **Delta** (printed by full_evaluation) - how much the frequency branch added over spatial alone
- **Gate entropy** - must be > 0.3 nats for gating experiments to be valid
- **Frozen vs fine-tuned** - if frozen delta > fine-tuned delta, the backbone was absorbing spectral information during fine-tuning
